# 101 · Data science perspective lab

Companion to [Data science perspective](https://leo-gan.github.io/GLD.SerializerBenchmark/theory/101/data-science-perspective/).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leo-gan/GLD.SerializerBenchmark/blob/master/docs/theory/notebooks/101/data_science_perspective.ipynb)

**Goal:** feel the difference between CSV/JSONL, pickle trust boundaries, and row vs columnar layout for analytics-style access.

> **Honesty banner:** sizes and timings here are **illustrative**. Suite [Results](https://leo-gan.github.io/GLD.SerializerBenchmark/) own harness truth.

Uses **stdlib only** so Colab runs without heavy deps. Optional cells detect `pyarrow` if present.



In [ ]:
import csv
import io
import json
import pickle
import struct
from pathlib import Path

# Tiny "lake" sample — wide enough to show column projection idea
ROWS = [
    {"event_id": i, "user": f"u{i%5}", "feature_a": i * 0.1, "feature_b": i % 3,
     "label": "pos" if i % 4 == 0 else "neg", "payload_note": "x" * 8}
    for i in range(100)
]
print(len(ROWS), "rows; columns:", list(ROWS[0]))



## CSV as a lossy social format



In [ ]:
buf = io.StringIO()
w = csv.DictWriter(buf, fieldnames=list(ROWS[0]))
w.writeheader()
w.writerows(ROWS)
csv_text = buf.getvalue()
print("CSV nbytes:", len(csv_text.encode()))
print(csv_text.splitlines()[0])
print(csv_text.splitlines()[1])
# Types are gone — everything is text on reload
buf.seek(0)
reread = list(csv.DictReader(buf))
print("feature_a type after CSV round-trip:", type(reread[0]["feature_a"]), reread[0]["feature_a"])



## JSONL — append-friendly landing zone



In [ ]:
jsonl = "\n".join(json.dumps(r, separators=(",", ":")) for r in ROWS).encode()
print("JSONL nbytes:", len(jsonl))
first = json.loads(jsonl.splitlines()[0])
print("feature_a type after JSONL:", type(first["feature_a"]), first["feature_a"])



## Pickle trust boundary

**Rule:** if bytes are not from a fully trusted source you control → **do not unpickle**.



In [ ]:
trusted = pickle.dumps(ROWS)
print("pickle nbytes:", len(trusted))
assert pickle.loads(trusted)[0]["event_id"] == 0

# Illustrative hostile pattern (do NOT run on untrusted bytes in production).
# pickle can invoke callables during load — we only show the *opcode surface*, not a live exploit.
print("pickle protocol opcodes (prefix):", trusted[:20])
print("OK: portable formats (JSONL/Parquet/Arrow) for multi-language or untrusted interchange")



## Row vs columnar access cost (toy model)

Simulates "read one column from a row store" vs "read one contiguous column buffer".



In [ ]:
# Row store: list of dicts (pointer-rich)
row_store = ROWS

# Column store: one list per column
col_store = {k: [r[k] for r in ROWS] for k in ROWS[0]}


def sum_feature_a_rows():
    return sum(r["feature_a"] for r in row_store)


def sum_feature_a_cols():
    return sum(col_store["feature_a"])


import timeit
from statistics import median

t_row = median(timeit.repeat(sum_feature_a_rows, number=2000, repeat=5))
t_col = median(timeit.repeat(sum_feature_a_cols, number=2000, repeat=5))
assert abs(sum_feature_a_rows() - sum_feature_a_cols()) < 1e-9
print(f"sum feature_a via rows: {t_row*1e3:.3f} ms median")
print(f"sum feature_a via cols: {t_col*1e3:.3f} ms median")
print("Columnar wins more as width grows and engines skip unread columns (Parquet/Arrow idea).")



## Optional: Parquet/Arrow if pyarrow is installed



In [ ]:
try:
    import pyarrow as pa
    import pyarrow.parquet as pq

    table = pa.Table.from_pylist(ROWS)
    sink = pa.BufferOutputStream()
    pq.write_table(table, sink, compression="zstd")
    parquet_bytes = sink.getvalue().to_pybytes()
    print("Parquet nbytes:", len(parquet_bytes))
    # Project one column only
    col_only = pq.read_table(pa.BufferReader(parquet_bytes), columns=["feature_a"])
    print("read feature_a only:", col_only.column(0).to_pylist()[:5], "…")
except ImportError:
    print("SKIP pyarrow — optional: pip install pyarrow")
    print("Pattern to remember: Parquet on disk / Arrow in memory between engines.")



## Takeaways

| Workload | Prefer |
|----------|--------|
| Human export | CSV (edge only) |
| Landing / logs | JSONL |
| Analytics scans | Columnar (Parquet-class) |
| Engine handoff | Arrow |
| Untrusted / multi-lang | Never pickle |

**Next:** [Engineering mini lab](./engineering_perspective.ipynb) · [Serialization 201](https://leo-gan.github.io/GLD.SerializerBenchmark/theory/201/)

